# Governance Table Setup

**Run once** to create all governance Delta tables with correct schemas and default configuration.

Requires: **Contributor** role + default Lakehouse attached.

### Tables created
| Table | Purpose |
|-------|---------|
| `cleanup_tracker` | Warning lifecycle state per item |
| `cleanup_audit_log` | Immutable log of all actions |
| `governance_config` | Configurable thresholds and rules |
| `email_outbox` | Generated emails queue for pipeline Outlook activity |


In [1]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from datetime import datetime, timezone

print("Creating governance Delta tables...")
print("=" * 50)


StatementMeta(, cd174e0f-dda1-4c9a-a006-02c677c877d2, 3, Finished, Available, Finished, False)

Creating governance Delta tables...


## 1. cleanup_tracker

In [2]:
schema_tracker = StructType([
    StructField("item_id", StringType(), False),
    StructField("item_name", StringType(), True),
    StructField("item_type", StringType(), True),
    StructField("owner_email", StringType(), True),
    StructField("cleanup_score", IntegerType(), True),
    StructField("first_flagged_date", StringType(), True),
    StructField("warning_count", IntegerType(), True),
    StructField("warning_1_date", StringType(), True),
    StructField("warning_2_date", StringType(), True),
    StructField("warning_3_date", StringType(), True),
    StructField("status", StringType(), True),
    StructField("deleted_date", StringType(), True),
    StructField("resolved_date", StringType(), True),
    StructField("exemption_reason", StringType(), True),
    StructField("last_updated", StringType(), True),
    StructField("pipeline_run_id", StringType(), True),
    StructField("workspace_id", StringType(), True),
])

df_empty = spark.createDataFrame([], schema_tracker)
df_empty.write.format("delta").mode("overwrite").saveAsTable("cleanup_tracker")
print("✔ cleanup_tracker created")


StatementMeta(, cd174e0f-dda1-4c9a-a006-02c677c877d2, 4, Finished, Available, Finished, False)

✔ cleanup_tracker created


## 2. cleanup_audit_log

In [3]:
schema_audit = StructType([
    StructField("audit_id", StringType(), False),
    StructField("timestamp", StringType(), True),
    StructField("pipeline_run_id", StringType(), True),
    StructField("item_id", StringType(), True),
    StructField("item_name", StringType(), True),
    StructField("item_type", StringType(), True),
    StructField("owner_email", StringType(), True),
    StructField("action", StringType(), True),
    StructField("detail", StringType(), True),
    StructField("workspace_id", StringType(), True),
])

df_empty = spark.createDataFrame([], schema_audit)
df_empty.write.format("delta").mode("overwrite").saveAsTable("cleanup_audit_log")
print("✔ cleanup_audit_log created")


StatementMeta(, cd174e0f-dda1-4c9a-a006-02c677c877d2, 5, Finished, Available, Finished, False)

✔ cleanup_audit_log created


## 3. email_outbox

In [4]:
schema_outbox = StructType([
    StructField("email_id", StringType(), False),
    StructField("pipeline_run_id", StringType(), True),
    StructField("email_type", StringType(), True),
    StructField("recipient", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("body_html", StringType(), True),
    StructField("status", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("workspace_id", StringType(), True),
])

df_empty = spark.createDataFrame([], schema_outbox)
df_empty.write.format("delta").mode("overwrite").saveAsTable("email_outbox")
print("✔ email_outbox created")


StatementMeta(, cd174e0f-dda1-4c9a-a006-02c677c877d2, 6, Finished, Available, Finished, False)

✔ email_outbox created


## 4. governance_config

In [5]:
# Default configuration values
config_defaults = [
    ("cleanup_score_threshold", "30",    "Min score to enter cleanup workflow"),
    ("stale_cutoff_days",       "90",    "Days to flag as stale"),
    ("warnings_before_delete",  "3",     "Warnings before auto-delete"),
    ("days_between_warnings",   "1",     "Min days between warnings (1=testing, 7=production)"),
    ("admin_email",             "SURYADEV.RATHORE@XEBIA.COM", "Dashboard report recipient"),
    ("pipeline_schedule",       "daily_9am", "When pipeline runs"),
    ("protected_types",         "Lakehouse,Warehouse,Environment,SQLEndpoint", "Types never auto-deleted"),
    ("protected_items",         "",       "Comma-separated item IDs never deleted"),
    ("enable_auto_delete",      "false",  "Master switch for deletion (start disabled)"),
    ("workspace_ids",           "3cb31722-1925-4f2b-93ca-139f4e816755", "Comma-separated workspace IDs"),
    ("logo_url",                "",       "Xebia logo URL or base64 for emails"),
    ("dry_run",                 "true",   "Log actions without executing (true for testing)"),
]

schema_config = StructType([
    StructField("config_key", StringType(), False),
    StructField("config_value", StringType(), True),
    StructField("description", StringType(), True),
])

df_config = spark.createDataFrame(config_defaults, schema_config)
df_config.write.format("delta").mode("overwrite").saveAsTable("governance_config")
print("✔ governance_config created with defaults:")
display(spark.sql("SELECT config_key, config_value FROM governance_config ORDER BY config_key"))


StatementMeta(, cd174e0f-dda1-4c9a-a006-02c677c877d2, 7, Finished, Available, Finished, False)

✔ governance_config created with defaults:


SynapseWidget(Synapse.DataFrame, e76c9c1c-4685-4d99-b003-7eb363cad5d0)

In [1]:
from PIL import Image
import io, base64

img = Image.open("/lakehouse/default/Files/Fabric_Monitoring/xebia_logo.png")

if img.mode != 'RGBA':
    img = img.convert('RGBA')

# Resize to 150px wide
ratio = 150 / img.size[0]
img_small = img.resize((150, int(img.size[1] * ratio)), Image.LANCZOS)

# White background version (logo is purple, needs light bg to be visible)
bg = Image.new('RGBA', img_small.size, (255, 255, 255, 255))
bg.paste(img_small, (0, 0), img_small)
img_final = bg.convert('RGB')

buf = io.BytesIO()
img_final.save(buf, format='PNG', optimize=True)
logo_data_uri = f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"

df_cfg = spark.sql("SELECT * FROM governance_config").toPandas()
df_cfg.loc[df_cfg["config_key"] == "logo_url", "config_value"] = logo_data_uri
spark.createDataFrame(df_cfg).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("governance_config")
print(f"✔ Real Xebia logo saved ({len(logo_data_uri)} chars)")

StatementMeta(, 58a6accc-cb2f-48e3-841b-1461e78a40f0, 3, Finished, Available, Finished, True)

✔ Real Xebia logo saved (12194 chars)


## 5. Verify all tables

In [6]:
%%sql
-- Verify all tables exist and show row counts
SELECT 'cleanup_tracker' AS table_name, COUNT(*) AS row_count FROM cleanup_tracker
UNION ALL
SELECT 'cleanup_audit_log', COUNT(*) FROM cleanup_audit_log
UNION ALL
SELECT 'email_outbox', COUNT(*) FROM email_outbox
UNION ALL
SELECT 'governance_config', COUNT(*) FROM governance_config
UNION ALL
SELECT 'workspace_inventory_snapshot', COUNT(*) FROM workspace_inventory_snapshot


StatementMeta(, cd174e0f-dda1-4c9a-a006-02c677c877d2, 8, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 2 fields>

## Done

All tables created. You can now run the **Governance_Tracker_Update** notebook.

To modify thresholds later:
```sql
UPDATE governance_config SET config_value = '50' WHERE config_key = 'cleanup_score_threshold'
```
